In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR100
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split, ConcatDataset
from torchvision import transforms
from loguru import logger
import matplotlib.image as mpimg
from sklearn.preprocessing import LabelEncoder
from nazi_symbols_classification.training.data_preparation import get_image_paths

In [2]:
dataset_path = "/home/zhiwei/Projects/nazi-symbols-classification/datasets/nazi-symbols-classification"

In [3]:
def load_labels_df(dataset_path, sub_dataset_name):
    image_paths = get_image_paths(dataset_path, sub_folders=(sub_dataset_name,))
    logger.info(f"Number of images in {sub_dataset_name} is {len(image_paths)}")
    training_image_paths = [image.removeprefix(f"os.path.join(dataset_path, sub_dataset_name)/") for image in image_paths]
    training_labels = [os.path.basename(os.path.dirname(image)) for image in training_image_paths]
    labels = pd.DataFrame(dict(path=training_image_paths, nazi_cls=training_labels))
    return labels

In [4]:
train_labels_df = load_labels_df(dataset_path, "train")
valid_labels_df = load_labels_df(dataset_path, "val")
test_labels_df = load_labels_df(dataset_path, "test")

2025-04-06 22:51:12.041 | INFO     | __main__:load_labels_df:3 - Number of images in train is 9009
2025-04-06 22:51:12.048 | INFO     | __main__:load_labels_df:3 - Number of images in val is 1934
2025-04-06 22:51:12.052 | INFO     | __main__:load_labels_df:3 - Number of images in test is 1944


In [5]:
le = LabelEncoder()

le.fit(train_labels_df.nazi_cls)
train_labels_df["nazi_cls_encoded"] = le.transform(train_labels_df.nazi_cls)
valid_labels_df["nazi_cls_encoded"] = le.transform(valid_labels_df.nazi_cls)
test_labels_df["nazi_cls_encoded"] = le.transform(test_labels_df.nazi_cls)

In [6]:
class ImageData(Dataset):
    def __init__(self, df, data_directory, transform):
        super().__init__()
        self.df = df
        self.data_directory = data_directory
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):       
        img_name = self.df.path[index]
        label = self.df.nazi_cls_encoded[index]
        
        img_path = os.path.join(self.data_directory, img_name)
        try:
            image = mpimg.imread(img_path)
        except Exception:
            if img_path.endswith(".png"):
                image = mpimg.imread(img_path, format="jpg")
            else:
                image = mpimg.imread(img_path, format="png")
        image = self.transform(image)
        return image, label

In [7]:
stats = ((0.5074,0.4867,0.4411),(0.2011,0.1987,0.2025))
data_transf = transforms.Compose([transforms.ToPILImage(), 
                                  transforms.Grayscale(num_output_channels=3),
                                  transforms.Resize((32, 32)), 
                                  transforms.ToTensor(),
                                  transforms.Normalize(*stats)])

def create_datasets(batch_size, dataset_path=dataset_path):
    train_data = ImageData(df = train_labels_df,
                           data_directory = os.path.join(dataset_path, 'train'),
                           transform = data_transf)
    train_loader = DataLoader(dataset = train_data, batch_size = batch_size, shuffle=True)
    
    valid_data = ImageData(df = valid_labels_df, 
                           data_directory = os.path.join(dataset_path, 'val'), 
                           transform = data_transf)
    valid_loader = DataLoader(dataset = valid_data, batch_size = batch_size, shuffle=False)

    test_data = ImageData(df = test_labels_df, 
                           data_directory = os.path.join(dataset_path, 'test'), 
                           transform = data_transf)
    test_loader = DataLoader(dataset = test_data, batch_size = batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

In [8]:
batch_size = 512

train_loader, valid_loader, test_loader = create_datasets(batch_size, dataset_path)

In [9]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def to_device(data,device):
    if isinstance(data,(list,tuple)):
        return [to_device(x,device) for x in data]
    return data.to(device,non_blocking=True)


class ToDeviceLoader:
    def __init__(self,data,device):
        self.data = data
        self.device = device
        
    def __iter__(self):
        for batch in self.data:
            yield to_device(batch,self.device)
            
    def __len__(self):
        return len(self.data)

In [10]:
device = get_device()
print(device)

train_dl = ToDeviceLoader(train_loader, device)
valid_dl = ToDeviceLoader(valid_loader, device)
test_dl = ToDeviceLoader(test_loader, device)

cuda


In [11]:
def accuracy(predicted, actual):
    _, predictions = torch.max(predicted, dim=1)
    return torch.tensor(torch.sum(predictions==actual).item()/len(predictions))

In [12]:
class BaseModel(nn.Module):
    def training_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        return loss
    
    def validation_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        acc = accuracy(out,labels)
        return {"val_loss":loss.detach(),"val_acc":acc}
    
    def validation_epoch_end(self,outputs):
        batch_losses = [loss["val_loss"] for loss in outputs]
        loss = torch.stack(batch_losses).mean()
        batch_accuracy = [accuracy["val_acc"] for accuracy in outputs]
        acc = torch.stack(batch_accuracy).mean()
        return {"val_loss":loss.item(),"val_acc":acc.item()}
    
    def epoch_end(self, epoch, result):
        print("Epoch [{}], last_lr: {:.5f}, train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}".format(
            epoch, result['lrs'][-1], result['train_loss'], result['val_loss'], result['val_acc']))

In [13]:
def conv_shortcut(in_channel, out_channel, stride):
    layers = [nn.Conv2d(in_channel, out_channel, kernel_size=(1,1), stride=(stride, stride)),
             nn.BatchNorm2d(out_channel)]
    return nn.Sequential(*layers)

def block(in_channel, out_channel, k_size,stride, conv=False):
    layers = None
    
    first_layers = [nn.Conv2d(in_channel,out_channel[0], kernel_size=(1,1),stride=(1,1)),
                    nn.BatchNorm2d(out_channel[0]),
                    nn.ReLU(inplace=True)]
    if conv:
        first_layers[0].stride=(stride,stride)
    
    second_layers = [nn.Conv2d(out_channel[0], out_channel[1], kernel_size=(k_size, k_size), stride=(1,1), padding=1),
                    nn.BatchNorm2d(out_channel[1])]

    layers = first_layers + second_layers
    
    return nn.Sequential(*layers)
    

class ResNet(BaseModel):
    
    def __init__(self, in_channels, num_classes):
        super().__init__()
        
        self.stg1 = nn.Sequential(
                                   nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=(3),
                                             stride=(1), padding=1),
                                   nn.BatchNorm2d(64),
                                   nn.ReLU(inplace=True),
                                   nn.MaxPool2d(kernel_size=3, stride=2))
        
        ##stage 2
        self.convShortcut2 = conv_shortcut(64,256,1)
        
        self.conv2 = block(64,[64,256],3,1,conv=True)
        self.ident2 = block(256,[64,256],3,1)

        
        ##stage 3
        self.convShortcut3 = conv_shortcut(256,512,2)
        
        self.conv3 = block(256,[128,512],3,2,conv=True)
        self.ident3 = block(512,[128,512],3,2)

        
        ##stage 4
        self.convShortcut4 = conv_shortcut(512,1024,2)
        
        self.conv4 = block(512,[256,1024],3,2,conv=True)
        self.ident4 = block(1024,[256,1024],3,2)
        
        
        ##Classify
        self.classifier = nn.Sequential(
                                       nn.AvgPool2d(kernel_size=(4)),
                                       nn.Flatten(),
                                       nn.Linear(1024, num_classes))
        
    def forward(self,inputs):
        out = self.stg1(inputs)
        
        #stage 2
        out = F.relu(self.conv2(out) + self.convShortcut2(out))
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        
        #stage3
        out = F.relu(self.conv3(out) + (self.convShortcut3(out)))
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        
        #stage4             
        out = F.relu(self.conv4(out) + (self.convShortcut4(out)))
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        
        #Classify
        out = self.classifier(out)#100x1024
        
        return out
        

In [14]:
model = ResNet(3,100)

In [15]:
model = to_device(model, device)

In [16]:
@torch.no_grad()
def evaluate(model,valid_dl):
    model.eval()
    outputs = [model.validation_step(batch) for batch in valid_dl]
    return model.validation_epoch_end(outputs)

In [17]:
from early_stopping_pytorch import EarlyStopping

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit (epochs, train_dl, valid_dl, model, optimizer, max_lr, weight_decay, scheduler, grad_clip=None, patience=10):
    torch.cuda.empty_cache()
    
    history = []
    
    optimizer = optimizer(model.parameters(), max_lr, weight_decay = weight_decay)
    
    scheduler = scheduler(optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_dl))

    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    for epoch in range(epochs):
        model.train()
        
        train_loss = []
        
        lrs = []
        
        for batch in train_dl:
            loss = model.training_step(batch)
            
            train_loss.append(loss)
            
            loss.backward()
            
            if grad_clip:
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            
            optimizer.step()
            optimizer.zero_grad()
            
            scheduler.step()
            lrs.append(get_lr(optimizer))
        result = evaluate(model, valid_dl)
        result["train_loss"] = torch.stack(train_loss).mean().item()
        result["lrs"] = lrs
        
        model.epoch_end(epoch,result)
        history.append(result)

        early_stopping(result["val_loss"], model)
        
        if early_stopping.early_stop:
            print("Early stopping")
            break
        
    return history

In [18]:
epochs = 100
optimizer = torch.optim.Adam
max_lr = 1e-3
grad_clip = 0.1
weight_decay = 1e-5
scheduler = torch.optim.lr_scheduler.OneCycleLR
patience = 10

In [19]:
%%time
history = fit(epochs=epochs, train_dl=train_dl, valid_dl=valid_dl, model=model, 
              optimizer=optimizer, max_lr=max_lr, grad_clip=grad_clip, patience=patience,
              weight_decay=weight_decay, scheduler=torch.optim.lr_scheduler.OneCycleLR)

Epoch [0], last_lr: 0.00004, train_loss: 5.3552, val_loss: 3.3954, val_acc: 0.4786
Validation loss decreased (inf --> 3.395445).  Saving model ...
Epoch [1], last_lr: 0.00005, train_loss: 2.8530, val_loss: 3.0922, val_acc: 0.1975
Validation loss decreased (3.395445 --> 3.092231).  Saving model ...
Epoch [2], last_lr: 0.00006, train_loss: 1.7437, val_loss: 2.2340, val_acc: 0.4786
Validation loss decreased (3.092231 --> 2.234003).  Saving model ...
Epoch [3], last_lr: 0.00008, train_loss: 1.2804, val_loss: 2.4741, val_acc: 0.3517
EarlyStopping counter: 1 out of 10
Epoch [4], last_lr: 0.00010, train_loss: 0.9142, val_loss: 2.7546, val_acc: 0.4847
EarlyStopping counter: 2 out of 10
Epoch [5], last_lr: 0.00013, train_loss: 0.6260, val_loss: 3.2124, val_acc: 0.4955
EarlyStopping counter: 3 out of 10
Epoch [6], last_lr: 0.00016, train_loss: 0.4649, val_loss: 3.6937, val_acc: 0.4893
EarlyStopping counter: 4 out of 10
Epoch [7], last_lr: 0.00020, train_loss: 0.3743, val_loss: 4.5886, val_acc: 0

In [18]:
model.load_state_dict(torch.load("best-model-resetnet-nazi-classification.pt", weights_only=True))
model.eval()

ResNet(
  (stg1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (convShortcut2): Sequential(
    (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv2): Sequential(
    (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (ident2): Sequential(
    (0): Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, aff

In [19]:
outputs = []
y_true = []
for batch in test_dl:
    images, labels = batch
    out = model(images)
    _, predicted = torch.max(out, 1)
    # print([le.classes_[predicted[i]] for i in range(len(images))])
    y_true += labels.cpu()
    outputs += predicted.cpu()

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(outputs, y_true))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         0
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         0
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         0
          11       0.00      0.00      0.00        15
          12       0.00      0.00      0.00         0
          13       0.00      0.00      0.00         0
          14       0.00      0.00      0.00         0
          15       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0
          17       0.00    

Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
